In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score
import joblib

# Dataset yükle
df = pd.read_csv('/Users/cemrebalci/Desktop/PhishAnalyzer/model/PhiUSIIL_Phishing_URL_Dataset.csv')

features = [
    'URLLength', 'IsHTTPS', 'NoOfSubDomain', 'IsDomainIP',
    'URLSimilarityIndex', 'TLDLegitimateProb', 'HasObfuscation',
    'NoOfObfuscatedChar', 'HasPasswordField', 'Bank', 'Pay',
    'Crypto', 'DegitRatioInURL', 'NoOfAmpersandInURL'
]

X = df[features]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

base_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    max_depth=20,
    min_samples_leaf=5
)

model_calibrated = CalibratedClassifierCV(base_model, cv=3, method='isotonic')
model_calibrated.fit(X_train, y_train)

y_pred = model_calibrated.predict(X_test)
print("Doğruluk:", accuracy_score(y_test, y_pred))

# Her iki klasöre de kaydet
joblib.dump(model_calibrated, '/Users/cemrebalci/Desktop/PhishAnalyzer/model/phishanalyzer_model.pkl')
joblib.dump(features, '/Users/cemrebalci/Desktop/PhishAnalyzer/model/phishanalyzer_features.pkl')
joblib.dump(model_calibrated, '/Users/cemrebalci/Desktop/PhishAnalyzer/backend/phishanalyzer/model/phishanalyzer_model.pkl')
joblib.dump(features, '/Users/cemrebalci/Desktop/PhishAnalyzer/backend/phishanalyzer/model/phishanalyzer_features.pkl')
print("✅ Model her iki klasöre de kaydedildi!")

Doğruluk: 0.9998727708390763
✅ Model her iki klasöre de kaydedildi!


In [2]:
import joblib
import pandas as pd

model = joblib.load('/Users/cemrebalci/Desktop/PhishAnalyzer/backend/phishanalyzer/model/phishanalyzer_model.pkl')
features = joblib.load('/Users/cemrebalci/Desktop/PhishAnalyzer/backend/phishanalyzer/model/phishanalyzer_features.pkl')

print("Model tipi:", type(model).__name__)

# fitgirl test
feats = {
    'URLLength': 45,
    'IsHTTPS': 1,
    'NoOfSubDomain': 2,
    'IsDomainIP': 0,
    'URLSimilarityIndex': 10.0,
    'TLDLegitimateProb': 0.5,
    'HasObfuscation': 0,
    'NoOfObfuscatedChar': 0,
    'HasPasswordField': 0,
    'Bank': 0,
    'Pay': 0,
    'Crypto': 0,
    'DegitRatioInURL': 0.0,
    'NoOfAmpersandInURL': 0,
}

X = pd.DataFrame([feats])[features]
prob = model.predict_proba(X)[0]
print(f"Phishing: %{prob[0]*100:.1f}")
print(f"Güvenli: %{prob[1]*100:.1f}")


Model tipi: CalibratedClassifierCV
Phishing: %100.0
Güvenli: %0.0


In [3]:
base_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    max_depth=10,        # Daha düşük derinlik
    min_samples_leaf=20, # Daha fazla minimum örnek
    min_samples_split=10
)

model_calibrated = CalibratedClassifierCV(
    base_model, 
    cv=5,              # Daha fazla fold
    method='sigmoid'   # isotonic yerine sigmoid dene
)
model_calibrated.fit(X_train, y_train)

# Test et
prob = model_calibrated.predict_proba(X_test)
print("Max confidence:", prob.max() * 100)
print("Min confidence:", prob.min() * 100)
print("Ortalama:", prob.max(axis=1).mean() * 100)

Max confidence: 99.9981965375102
Min confidence: 0.0018034624897959035
Ortalama: 99.98466664032895


In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score
import joblib

df = pd.read_csv('/Users/cemrebalci/Desktop/PhishAnalyzer/model/PhiUSIIL_Phishing_URL_Dataset.csv')

# Extra güvenli siteler ekle
extra_safe_urls = [
    'https://eksisozluk.com',
    'https://eksisozluk.com/entry/123',
    'https://hepsiburada.com',
    'https://trendyol.com',
    'https://sahibinden.com',
    'https://hurriyet.com.tr',
    'https://milliyet.com.tr',
    'https://ntv.com.tr',
    'https://fitgirl-repacks.site',
    'https://stackoverflow.com',
    'https://medium.com',
    'https://twitter.com',
    'https://linkedin.com',
    'https://instagram.com',
    'https://twitch.tv',
    'https://spotify.com',
    'https://netflix.com',
    'https://zoom.us',
    'https://slack.com',
    'https://www.google.com',
    'https://google.com',
    'https://mail.google.com',
    'https://drive.google.com',
    'https://www.youtube.com',
    'https://www.facebook.com',
    'https://www.amazon.com',
    'https://www.microsoft.com',
    'https://github.com',
    'https://wikipedia.org',
    'https://reddit.com',
]

extra_rows = []
for url in extra_safe_urls:
    extra_rows.append({
        'URLLength': len(url),
        'IsHTTPS': 1,
        'NoOfSubDomain': url.count('.') - 1,
        'IsDomainIP': 0,
        'HasObfuscation': 0,
        'NoOfObfuscatedChar': 0,
        'HasPasswordField': 0,
        'Bank': 0,
        'Pay': 0,
        'Crypto': 0,
        'DegitRatioInURL': sum(c.isdigit() for c in url) / len(url),
        'NoOfAmpersandInURL': 0,
        'URLSimilarityIndex': 5.0,
        'TLDLegitimateProb': 0.9,
        'label': 1
    })

extra_df = pd.DataFrame(extra_rows)
df_extended = pd.concat([df, extra_df], ignore_index=True)
print("Genişletilmiş dataset:", len(df_extended))

features = [
    'URLLength', 'IsHTTPS', 'NoOfSubDomain', 'IsDomainIP',
    'URLSimilarityIndex', 'TLDLegitimateProb', 'HasObfuscation',
    'NoOfObfuscatedChar', 'HasPasswordField', 'Bank', 'Pay',
    'Crypto', 'DegitRatioInURL', 'NoOfAmpersandInURL'
]

X = df_extended[features]
y = df_extended['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

base_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    max_depth=20,
    min_samples_leaf=5
)

model_calibrated = CalibratedClassifierCV(base_model, cv=3, method='isotonic')
model_calibrated.fit(X_train, y_train)

y_pred = model_calibrated.predict(X_test)
print("Doğruluk:", accuracy_score(y_test, y_pred))

# Kaydet
joblib.dump(model_calibrated, '/Users/cemrebalci/Desktop/PhishAnalyzer/model/phishanalyzer_model.pkl')
joblib.dump(features, '/Users/cemrebalci/Desktop/PhishAnalyzer/model/phishanalyzer_features.pkl')
joblib.dump(model_calibrated, '/Users/cemrebalci/Desktop/PhishAnalyzer/backend/phishanalyzer/model/phishanalyzer_model.pkl')
joblib.dump(features, '/Users/cemrebalci/Desktop/PhishAnalyzer/backend/phishanalyzer/model/phishanalyzer_features.pkl')
print("✅ Model kaydedildi!")

Genişletilmiş dataset: 235825
Doğruluk: 0.9998939891868971
✅ Model kaydedildi!
